In [ ]:
!pip install sunpy[all]
!pip install batman-package
!pip install numpy scipy matplotlib astropy sunpy drms

In [ ]:
### main code *****

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sunpy.net import Fido, attrs as a
from astropy.io import fits
from astropy.time import Time
from astropy import units as u
from scipy.signal import find_peaks
from scipy.interpolate import PchipInterpolator

# =========================================================
# 1. DOWNLOAD DATA (5 min cadence)
# =========================================================
tstart = "2012-06-02T12:00:00"
tend   = "2012-06-09T00:00:00"

print("Downloading SDO data (5 min cadence)...")
result = Fido.search(
    a.Time(tstart, tend),
    a.Instrument.hmi,
    a.Physobs.intensity,
    a.Sample(5 * u.minute)
)
files = sorted(Fido.fetch(result))

# =========================================================
# 2. EXTRACT FLUX
# =========================================================
times, fluxes = [], []

print("Extracting flux...")
for f in files:
    try:
        with fits.open(f) as hdul:
            idx = 1 if len(hdul) > 1 else 0
            header = hdul[idx].header
            data = hdul[idx].data

            if data is None: continue

            t_str = header.get("DATE-OBS", header.get("T_OBS"))
            exptime = header.get('EXPTIME', 1.0)
            if exptime <= 0: exptime = 1.0

            val = np.nansum(data) / exptime
            times.append(Time(t_str).datetime)
            fluxes.append(val)
    except:
        continue

times = np.array(times)
fluxes = np.array(fluxes)
sort_idx = np.argsort(times)
times = times[sort_idx]
fluxes = fluxes[sort_idx]

# Normalize raw data to median=1
fluxes = fluxes / np.median(fluxes)

# =========================================================
# 3. ENVELOPE INTERPOLATION (THE FIX)
# =========================================================
t_nums = mdates.date2num(times)

# 1. Find Peaks
# For 15-min cadence, 1 day = 288 points.
# so distance=250 perfectly isolates the 24-hour orbital wobble.
peaks, _ = find_peaks(fluxes, distance=250)

# 2. Connect the Dots (Interpolation instead of Polynomial)
# We use PCHIP interpolation because it doesn't "overshoot" or oscillate like normal Splines
envelope_maker = PchipInterpolator(t_nums[peaks], fluxes[peaks])

# 3. Generate the Envelope
# We can interpolate BETWEEN the first and last peak.
# Data outside this range will be ignored to prevent errors.
valid_mask = (t_nums >= t_nums[peaks[0]]) & (t_nums <= t_nums[peaks[-1]])

fitted_times = times[valid_mask]
fitted_fluxes = fluxes[valid_mask]
fitted_t_nums = t_nums[valid_mask]

# Generate the perfect envelope for the valid range
baseline_curve = envelope_maker(fitted_t_nums)

# =========================================================
# 4. FLATTEN
# =========================================================
# Divide Raw by Envelope
flat_flux = fitted_fluxes / baseline_curve

# =========================================================
# 5. PLOT
# =========================================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# --- TOP PLOT ---
ax1.plot(times, fluxes, color='#1f77b4', lw=1.5, alpha=0.6, label='Raw Flux')
ax1.plot(times[peaks], fluxes[peaks], 'rx', markersize=8, markeredgewidth=2, label='Detected Peaks')
# Plot the envelope only where valid
ax1.plot(fitted_times, baseline_curve, color='k', lw=2, linestyle='--', label='PCHIP Envelope')
ax1.set_ylabel("Normalized Flux")
ax1.legend(loc='lower right')
ax1.set_title("1. Envelope (PCHIP Interpolation)")
ax1.grid(True, alpha=0.3)

# --- BOTTOM PLOT ---
ax2.plot(fitted_times, flat_flux, color='#1f77b4', lw=1.5, label='Flattened Flux')
ax2.axhline(1.0, color='gray', linestyle='--', lw=1.5)

# Highlight Transit
transit_start = mdates.date2num(Time("2012-06-05 22:00").datetime)
transit_end   = mdates.date2num(Time("2012-06-06 05:00").datetime)
ax2.axvspan(transit_start, transit_end, color='yellow', alpha=0.3, label='Transit Window')

ax2.set_ylabel("Normalized Flux")
ax2.set_xlabel("Date (June 2012)")
ax2.set_title("2. Result: Perfectly Flat Peaks")
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d'))
ax2.xaxis.set_major_locator(mdates.DayLocator())

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sunpy.net import Fido, attrs as a
from astropy.io import fits
from astropy.time import Time
from astropy import units as u
import batman  # The library used in the paper

# =========================================================
# 1. DOWNLOAD DATA (EXACT WINDOWS FROM PAPER)
# =========================================================
# The paper uses a "Reference Segment" (June 3-4) and "Transit Segment" (June 5-6)
# We use 2-minute cadence (Paper used 45s, but 2m is faster for you and looks same)
print("Downloading Reference & Transit Data...")

# Window 1: Reference (Quiet Day) - June 3 13:29 to June 4 13:29
result_ref = Fido.search(
    a.Time("2012-06-03T13:29:00", "2012-06-04T13:29:00"),
    a.Instrument.hmi, a.Physobs.intensity, a.Sample(1.5 * u.minute)
)

# Window 2: Transit (Event Day) - June 5 13:29 to June 6 13:29
result_trans = Fido.search(
    a.Time("2012-06-05T13:29:00", "2012-06-06T13:29:00"),
    a.Instrument.hmi, a.Physobs.intensity, a.Sample(1.5 * u.minute)
)

files_ref = sorted(Fido.fetch(result_ref))
files_trans = sorted(Fido.fetch(result_trans))

# =========================================================
# 2. HELPER FUNCTION TO EXTRACT FLUX
# =========================================================
def get_flux(files):
    times, fluxes = [], []
    for f in files:
        try:
            with fits.open(f) as hdul:
                idx = 1 if len(hdul) > 1 else 0
                header = hdul[idx].header
                data = hdul[idx].data
                if data is None: continue

                # Normalize by Exposure Time (Crucial for HMI)
                exptime = header.get('EXPTIME', 1.0)
                if exptime <= 0: exptime = 1.0

                val = np.nansum(data) / exptime
                t_rec = Time(header.get("DATE-OBS", header.get("T_OBS"))).datetime

                times.append(t_rec)
                fluxes.append(val)
        except:
            continue
    return np.array(times), np.array(fluxes)

print("Processing Fluxes...")
t_ref, f_ref = get_flux(files_ref)
t_trans, f_trans = get_flux(files_trans)

# =========================================================
# 3. STEP 1: MODEL THE 24H WOBBLE (Paper Section 3)
# =========================================================
# The paper fits a "16-order polynomial" to the reference segment.

# Convert both time series to "Hours from Start of Window" (0 to 24h)
# This aligns the orbital phases of the two days.
t_ref_hours = np.array([(t - t_ref[0]).total_seconds()/3600.0 for t in t_ref])
t_trans_hours = np.array([(t - t_trans[0]).total_seconds()/3600.0 for t in t_trans])

# Normalize Reference Flux to median=1
f_ref_norm = f_ref / np.median(f_ref)

# Fit 16th order polynomial to Reference Day
coeffs_ref = np.polyfit(t_ref_hours, f_ref_norm, 16)
wobble_model_poly = np.poly1d(coeffs_ref)

# Evaluate this wobble model at the TRANSIT times
predicted_wobble = wobble_model_poly(t_trans_hours)

# =========================================================
# 4. STEP 2: CORRECT THE TRANSIT DATA
# =========================================================
# Normalize Transit Flux
f_trans_norm = f_trans / np.median(f_trans)

# Divide Transit Data by the Predicted Wobble [cite: 202]
detrended_flux = f_trans_norm / predicted_wobble

# --- Final Polish (Remove Slow Linear Drift) ---
# The paper mentions a residual "slow drift" removed by a low-order fit.
# We fit a 2nd order poly to the wings (outside transit)
t_mid = Time("2012-06-06T01:29:35").datetime
t_days_from_mid = np.array([(t - t_mid).total_seconds()/86400.0 for t in t_trans])

# Mask Transit (Approx -0.15 to +0.15 days)
mask_wings = (t_days_from_mid < -0.16) | (t_days_from_mid > 0.16)

# Fit wings
coeffs_polish = np.polyfit(t_days_from_mid[mask_wings], detrended_flux[mask_wings], 2)
polish_model = np.polyval(coeffs_polish, t_days_from_mid)

# Final Flux
final_flux = detrended_flux / polish_model
final_flux /= np.median(final_flux[mask_wings]) # Ensure exactly 1.0

# =========================================================
# 5. EXTRACT EMPIRICAL DEPTH (No Batman!)
# =========================================================
# We find the depth by taking the median of the data points at the very
# bottom of the transit "U" (within 0.05 days of mid-transit)
mid_transit_mask = np.abs(t_days_from_mid) < 0.05
transit_depth = 1.0 - np.median(final_flux[mid_transit_mask])

print(f"Observed Transit Depth (δ): {transit_depth * 1e6:.0f} ppm")

# =========================================================
# 6. CALCULATE RADIUS WITH ANGULAR CORRECTION
# =========================================================
# Orbital parameters from Pyne et al. 2026
a_venus = 0.723  # AU (Venus semi-major axis)
d_earth = 1.000  # AU (Earth distance to Sun)
R_sun_km = 695900 # km

# 1. The Naive "Exoplanet" Calculation (Assuming d_earth is infinity)
Rp_Rs_naive = np.sqrt(transit_depth)
Rp_naive_km = Rp_Rs_naive * R_sun_km

# 2. The Angular Corrected Calculation (Pyne et al. 2026, Eq A3)
# Because Earth is so close, the apparent depth is heavily inflated.
correction_factor = (1.0 - (a_venus / d_earth))
Rp_Rs_corrected = np.sqrt(transit_depth) * correction_factor
Rp_corrected_km = Rp_Rs_corrected * R_sun_km

print(f"\n--- RADIUS EXTRACTION ---")
print(f"Naive Exoplanet Assumption: {Rp_naive_km:,.0f} km (Massively inflated!)")
print(f"Angular Corrected Radius:  {Rp_corrected_km:,.0f} km (Close to true ~6051 km)")

# =========================================================
# 7. PLOT THE DATA AND EXTRACTED DEPTH
# =========================================================
plt.figure(figsize=(10, 6))

# Plot the real SDO Data
plt.plot(t_days_from_mid, final_flux, '.', color='#bcbd22', markersize=5, alpha=0.7, label='Extracted SDO Data')

# Plot our extracted empirical depth as a red dashed line
plt.axhline(1.0 - transit_depth, color='red', linestyle='--', lw=2, label=f'Empirical Depth ({transit_depth*1e6:.0f} ppm)')
plt.axhline(1.0, color='black', linestyle='-', alpha=0.5, lw=1)

plt.xlabel("Time from Mid-transit (Days)", fontsize=12, fontname="Serif")
plt.ylabel("Normalized Flux", fontsize=12, fontname="Serif")
plt.title("Venus Transit: Empirical Depth ", fontsize=14)

plt.xlim(-0.35, 0.35)
plt.ylim(0.9985, 1.0005)
plt.tick_params(direction='in', length=6, width=1.5, top=True, right=True)

plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# =========================================================
# 8. VISUALIZE THE 16TH-ORDER WOBBLE MODEL
# =========================================================
plt.figure(figsize=(10, 4))

# Scatter plot of the actual normalized Reference Day data
plt.plot(t_ref_hours, f_ref_norm, '.', color='gray', alpha=0.5, label='Reference Data (June 3-4)')

# Generate a smooth line using your 16th-order model
smooth_hours = np.linspace(0, 24, 500)
plt.plot(smooth_hours, wobble_model_poly(smooth_hours), 'r-', lw=3, label='16th-Order Wobble Fit')

plt.title("SDO 24-Hour Instrumental Wobble Model")
plt.xlabel("Hours from Start of Reference Window")
plt.ylabel("Normalized Flux")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()